# 3d-point-cloud (unet branch) -- Colab training

Trains the sparse 3D U-Net backbone experiment (`configs/exp_sparse_unet.yaml`, experiment 5 in the dense/sparse/SlotFormer-depth comparison -- see `README.md`'s "Backbone registry" section), and optionally the other 4 experiments via `run_all_experiments.py`.

**Before running for real**: `exp_sparse_unet.yaml`'s `BATCH_SIZE: 8` is an *unmeasured* starting guess (see its own comment) -- run the "quick batch-size safety check" section below first and adjust it if needed. Every other `exp_*.yaml` was tuned this same way (see their comments for the numbers measured on an RTX 2070) -- Colab GPUs (T4/A100/etc) are different hardware, so don't assume those numbers transfer.

Runtime -> Change runtime type -> GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone and install

The model code lives on the `unet` branch specifically -- a plain `git clone` without `-b unet` checks out `main` instead and won't have `backbone3d_unet.py`/`exp_sparse_unet.yaml`.

In [ ]:
import os

# Safe to re-run this cell any number of times, from any state (fresh /content,
# already inside the repo, or already cloned but not cd'ed in) -- avoids the
# nested-clone trap (3d-point-cloud/3d-point-cloud/...) that a plain
# `!git clone && %cd` gives you if you re-run this cell after the first `%cd`
# already moved you inside the repo.
if os.path.basename(os.getcwd()) == "3d-point-cloud" and os.path.exists("train.py"):
    print("already inside 3d-point-cloud/ -- nothing to do")
else:
    if not os.path.isdir("3d-point-cloud"):
        !git clone -b unet https://github.com/izione/3d-point-cloud.git
    else:
        print("3d-point-cloud/ already exists here -- skipping clone")
    %cd 3d-point-cloud

!pip install -q -r requirements.txt

Optional: spconv accelerates the sparse backbones if it installs cleanly for this Colab image's CUDA version (`nvcc --version` or `!nvidia-smi` shows it) -- everything works without it too (pure-PyTorch fallback, `models/backbone3d_auto.py` probes automatically), so skip this cell if the install fails or you're not sure which `cuXXX` tag matches.

In [ ]:
# !pip install -q spconv-cu126   # pick the cuXXX tag matching this runtime's CUDA (see https://github.com/traveller59/spconv)

## 2. Get the dataset

Downloads `dataset.zip` straight from a Google Drive share link and unzips it onto the Colab VM's **local disk** (`/content/dataset`) -- deliberately not the Drive-mounted path: `SonarDiverDataset` reads tens of thousands of individual small `.bin`/`.json` files per epoch, and Drive's network filesystem is much slower than local disk for that access pattern (also skips re-uploading the zip from your own machine, which is what this is replacing).

Set `DATASET_ZIP_SHARE_URL` to the Drive share link ("Anyone with the link" viewer access, or you'll get a permission error) -- `gdown` accepts the ordinary `.../file/d/<id>/view?usp=sharing` link as-is (`fuzzy=True` extracts the file id itself).

In [ ]:
DATASET_ZIP_SHARE_URL = "https://drive.google.com/file/d/1JTnVQ1c25z2MfJQUvIHz4dgyRsFiPLm_/view?usp=drive_link"

In [ ]:
!pip install -q gdown
!gdown --fuzzy "{DATASET_ZIP_SHARE_URL}" -O /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/dataset_extracted
!ls /content/dataset_extracted

The next cell finds `DATASET_ROOT` itself (some `dataset.zip` files have the `PersonX/` folders directly at the top level, others have them nested one level down inside a wrapper folder e.g. `dataset/` -- both are common depending on how the zip was made) by walking down as long as there's exactly one subfolder and no `Person*` folder yet. No manual path-guessing needed even if you re-zip the dataset differently later.

In [ ]:
import os
import yaml

# Auto-detect DATASET_ROOT: descend past wrapper folders (e.g. dataset.zip's
# contents landing in /content/dataset_extracted/dataset/PersonX/... instead of
# directly in /content/dataset_extracted/PersonX/...) until we find a level
# that actually contains PersonX folders, or hit a level with more than one
# subfolder (ambiguous -- stop and let you set DATASET_ROOT by hand instead).
DATASET_ROOT = "/content/dataset_extracted"
while True:
    entries = [e for e in os.listdir(DATASET_ROOT) if not e.startswith(".")]
    if any(e.startswith("Person") for e in entries):
        break
    subdirs = [e for e in entries if os.path.isdir(os.path.join(DATASET_ROOT, e))]
    if len(subdirs) != 1:
        raise RuntimeError(
            f"couldn't auto-detect DATASET_ROOT under {DATASET_ROOT} (found {entries!r}, "
            f"expected exactly one wrapper folder or PersonX/ directories) -- set DATASET_ROOT by hand instead"
        )
    DATASET_ROOT = os.path.join(DATASET_ROOT, subdirs[0])

print("DATASET_ROOT ->", DATASET_ROOT)

# Point DATA.ROOT at it instead of hand-editing the yaml.
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["DATA"]["ROOT"] = DATASET_ROOT
with open("configs/default.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("DATA.ROOT ->", DATASET_ROOT)

No shareable zip link yet, or prefer the manual route? Upload the dataset to Drive yourself and mount it instead -- slower per-epoch (Drive-mounted reads), but no zip/link needed:

```python
from google.colab import drive
drive.mount('/content/drive')
DATASET_ROOT = "/content/drive/MyDrive/dataset"  # wherever you uploaded PersonX/scene_XXXX/
# ... then run the DATA.ROOT-patching cell above with this DATASET_ROOT
```

## 3. Mount Drive (for checkpoint persistence)

Separate from the dataset above -- this is just so training checkpoints survive a Colab disconnect (step 6 writes `--ckpt_dir` here).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Sanity check (synthetic data, no dataset needed)

Confirms every `configs/exp_*.yaml` -- including `exp_sparse_unet.yaml` -- constructs, runs forward/backward, and produces finite losses, before touching the real dataset.

In [ ]:
!python smoke_test.py

## 5. Quick batch-size AND speed safety check (real data, sparse_unet)

Runs a handful of real steps and reports peak GPU memory **and** an estimated epoch/full-run time -- same check every other `exp_*.yaml`'s `BATCH_SIZE` comment cites a number from, plus timing this repo hasn't measured for `sparse_unet` at all yet. The U-Net backbone's decoder restores full input resolution before the head (`total_stride=1`, vs. the other backbones' `total_stride=2`) -- the head/assigner/loss end up processing far more active voxels per step than the encoder-only experiments, on top of the encoder+decoder having roughly 2x the conv layers -- so don't assume this trains at a similar wall-clock pace to `exp_sparse_slotformer_3l` etc. just because the config file looks similar.

If `peak reserved` is above ~80% of this runtime's total GPU memory, lower `BATCH_SIZE` in `configs/exp_sparse_unet.yaml` (and scale `LR` proportionally -- see the config's own comment for the linear-scaling convention this project uses) and re-run this cell. If the estimated full-run time is impractical, the levers to shrink it (roughly in order of impact) are: fewer/narrower `STAGE_CHANNELS`, fewer `NUM_EPOCHS`, or accepting a coarser `DECODER_OUT_CHANNELS` -- all in `configs/exp_sparse_unet.yaml`.

In [ ]:
import time

import torch
from torch.utils.data import DataLoader

from config_utils import load_config
from data.dataset import SonarDiverDataset, collate_fn
from models.detector import DiverDetector

BATCH_SIZE_TO_TEST = None   # None = read OPTIMIZATION.BATCH_SIZE from the config below; set an int here to override
N_WARMUP = 5    # excluded from timing (first-call overhead: cuDNN autotune, allocator growth, etc.)
N_TIMED = 30

device = torch.device("cuda")
cfg = load_config("configs/exp_sparse_unet.yaml")
if BATCH_SIZE_TO_TEST is None:
    BATCH_SIZE_TO_TEST = cfg["OPTIMIZATION"]["BATCH_SIZE"]
ds = SonarDiverDataset(cfg, "train")
loader = DataLoader(ds, batch_size=BATCH_SIZE_TO_TEST, shuffle=True, collate_fn=collate_fn, drop_last=True)
model = DiverDetector(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
steps_per_epoch = len(ds) // BATCH_SIZE_TO_TEST
print(f"train frames: {len(ds)}  steps/epoch at batch={BATCH_SIZE_TO_TEST}: {steps_per_epoch}")

def run_one_step(it):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    losses, pred, stem_coords, assign_result = model.loss(batch, device)
    optimizer.zero_grad()
    losses["total"].backward()
    optimizer.step()
    return it, stem_coords.shape[0]

it = iter(loader)
torch.cuda.reset_peak_memory_stats()
for _ in range(N_WARMUP):
    it, _ = run_one_step(it)

torch.cuda.synchronize()
t0 = time.perf_counter()
last_n_voxels = None
for _ in range(N_TIMED):
    it, last_n_voxels = run_one_step(it)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

sec_per_step = elapsed / N_TIMED
reserved = torch.cuda.max_memory_reserved() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
epoch_min = sec_per_step * steps_per_epoch / 60
num_epochs = cfg["OPTIMIZATION"]["NUM_EPOCHS"]

print(f"peak_reserved={reserved:.2f} GiB / {total_mem:.1f} GiB ({100*reserved/total_mem:.0f}%)  "
      f"stem_voxels(last step)={last_n_voxels}")
print(f"{sec_per_step*1000:.0f} ms/step  ->  ~{epoch_min:.1f} min/epoch  ->  "
      f"~{epoch_min*num_epochs/60:.1f} hours for all {num_epochs} epochs (NUM_EPOCHS in the config)")

del model, optimizer, loader, ds
torch.cuda.empty_cache()

## 6. Train

Checkpoints/logs go straight to Drive (`--ckpt_dir`) so a Colab disconnect mid-epoch doesn't lose progress -- `CKPT_EVERY_N_EPOCHS`/`CKPT_EVERY_N_STEPS` in the config control how often that happens.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/3d-point-cloud-checkpoints/sparse_unet"

!python train.py --config configs/exp_sparse_unet.yaml --ckpt_dir "{CKPT_DIR}" --exp_name sparse_unet

Resume after a disconnect (picks up the schedule/step count from the checkpoint -- see `train.py`'s own `--resume` handling):

In [ ]:
# !python train.py --config configs/exp_sparse_unet.yaml --ckpt_dir "{CKPT_DIR}" --exp_name sparse_unet --resume "{CKPT_DIR}/sparse_unet_last.pth"

## 7. Optional: run all 5 experiments back-to-back

`run_all_experiments.py` runs dense / sparse-only / sparse+SlotFormer(3L) / sparse+SlotFormer(6L) / sparse U-Net in sequence, each into its own subdirectory under `--ckpt_dir`'s parent via its own `<name>` (see the script's docstring). Point `--ckpt_dir` names at Drive the same way as above if you use this instead of the single `train.py` call in step 6.

In [ ]:
# !python run_all_experiments.py --only sparse_unet
# !python run_all_experiments.py   # all 5 -- see BATCH_SIZE caveats above for each config first

## 8. Evaluate

```bash
!python test.py --checkpoint "{CKPT_DIR}/sparse_unet_last.pth" --split test
```

See `README.md`'s "Test / evaluate" section for `--pr_curve_out` (PR-curve-per-IoU-threshold plot) and `eval_pr_comparison.py` for comparing multiple checkpoints (e.g. this U-Net run vs. `exp_sparse_slotformer_3l`) on one figure.